# FlytBase Hackathon — Kaggle Starter

Kaggle counterpart of `notebooks/colab_starter.ipynb`. Same purpose (GPU check, core deps,
two model smoke tests), adapted for Kaggle's persistence model: `/kaggle/working/` persists
for the life of a session automatically, no Drive-style mount needed. Given Kaggle sessions
run 9-12 hours interactively, one long session likely covers the whole hackathon build
window - cross-session caching (via a Kaggle Dataset/Model attached as input) is only worth
adding if we actually hit a session restart tomorrow.

Pushed/pulled via `kaggle kernels push|pull|status|output` from the local repo - see
`CHECKLIST.md` section 6 for the sync workflow.

## 1. GPU check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}:", torch.cuda.get_device_name(i),
              round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1), "GB")

## 1b. Pin torch/torchvision before installing anything else

**Learned the hard way tonight**: an unconstrained `pip install` of `accelerate`,
`supervision`, or transformers-from-source can silently upgrade `torch` to a build whose
compiled CUDA kernels don't match this container's actual GPU - it loads and runs fine right
up until the first real inference call, which crashes with
`AcceleratorError: CUDA error: no kernel image is available for execution on the device`.
No warning at install time, unlike the pandas issue - it fails deep inside a forward pass.

Fix: capture whatever torch/torchvision are *already* correctly matched to this GPU right
now, and pin them explicitly in every subsequent install so pip's resolver can never touch
them.

In [ ]:
import torch
TORCH_PIN = torch.__version__
try:
    import torchvision
    TORCHVISION_PIN = torchvision.__version__
except ImportError:
    TORCHVISION_PIN = None

pins = f'"torch=={TORCH_PIN}"'
if TORCHVISION_PIN:
    pins += f' "torchvision=={TORCHVISION_PIN}"'

print("Pinning for all subsequent installs:", pins)

## 2. Model cache

`/kaggle/working/` persists for the session and is what gets saved if you commit
("Save Version") the notebook with output enabled.

In [ ]:
import os
CACHE_DIR = "/kaggle/working/model_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_HUB_CACHE"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
print("Model cache ->", CACHE_DIR)

## 3. Core dependencies

Same lean set as the Colab notebook. **Deliberately no `-U` and no explicit `pandas`/
`pyarrow`** - Kaggle's base image, like Colab's, pins pandas/numpy versions that RAPIDS
cuDF and other preinstalled packages depend on; forcing an upgrade broke Colab's equivalent
stack earlier tonight, so applying the same caution here proactively. `{pins}` from the
previous cell locks torch/torchvision the same way.

In [ ]:
!pip install -q accelerate faiss-cpu decord av scenedetect supervision \
    rank_bm25 rapidocr-onnxruntime pillow opencv-python-headless {pins}

## 4. Transformers — built from source (for Qwen3-VL support)

**Not `--no-deps`** (tried that, it broke: transformers-main wants `tokenizers>=0.23.1,<0.24.0`
and without letting pip resolve that, it left the older `tokenizers==0.22.2` in place, which
then fails at import time). Instead, pass the same `{pins}` used elsewhere - pip resolves
everything else (tokenizers, huggingface_hub, safetensors...) normally to satisfy what
transformers actually needs, while torch/torchvision stay locked to the exact already-working
versions no matter what.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers {pins}

## 5. Smoke test 1 — SigLIP2 (retrieval embeddings)

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor
from transformers.image_utils import load_image

SIGLIP_CKPT = "google/siglip2-so400m-patch14-384"
siglip_model = AutoModel.from_pretrained(SIGLIP_CKPT, device_map="auto").eval()
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_CKPT)

image = load_image(
    "https://huggingface.co/datasets/merve/coco/resolve/main/val2017/000000000285.jpg"
)
inputs = siglip_processor(images=[image], return_tensors="pt").to(siglip_model.device)
with torch.no_grad():
    image_embedding = siglip_model.get_image_features(**inputs)

print("SigLIP2 loaded OK. Embedding shape:", image_embedding.shape)

## 6. Smoke test 2 — Qwen3-VL-4B-Instruct (captioning)

Same caveat as the Colab notebook: if `Qwen3VLForConditionalGeneration` has been renamed by
the time this runs, check the model's HuggingFace "Use this model" snippet directly.

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor as QwenProcessor

QWEN_CKPT = "Qwen/Qwen3-VL-4B-Instruct"
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_CKPT, dtype="auto", device_map="auto", trust_remote_code=True
)
qwen_processor = QwenProcessor.from_pretrained(QWEN_CKPT, trust_remote_code=True)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image in one sentence."},
        ],
    }
]

qwen_inputs = qwen_processor.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt",
).to(qwen_model.device)

generated_ids = qwen_model.generate(**qwen_inputs, max_new_tokens=64)
output_text = qwen_processor.batch_decode(
    generated_ids[:, qwen_inputs["input_ids"].shape[1]:], skip_special_tokens=True
)
print("Qwen3-VL-4B loaded OK. Caption:", output_text[0])

## Done

If both smoke tests printed output with no errors, this Kaggle account/session is ready for
tomorrow's real pipeline work.